# Gold Play State — Possession Zone Sequences

Loads data from `bundesliga-2022-2023.batch.silver_positions` and produces two gold tables:

* **`gold_possessions_all`** — Team possession sequences across all play states with cumulative time
* **`gold_possession_zones`** — Zone-level possession sequences with team and opponent metrics (struct fields), `possession_id` and `stage_id`

Each match has \~1,300 zone sequences across 7 matches (\~9,095 total). Sequences are identified via gaps-and-islands on `possession_zone` + `team_id`, then aggregated with cumulative scores, ball distance metrics, and offside line averages.

In [0]:
# ── Load silver_positions and show summary ──

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table('`bundesliga-2022-2023`.batch.silver_positions')

print(f"Table: `bundesliga-2022-2023`.batch.silver_positions")
print(f"  Columns: {len(silver_df.columns)}")
print(f"  Rows: {silver_df.count():,}")
print(f"  Schema:")
for f in silver_df.schema.fields:
    print(f"    {f.name}: {f.dataType}")

print("\n=== Rows per team ===")
silver_df.groupBy("team_id").count().orderBy("team_id").show()

## Gold Possession Zones — Transformation Steps

Input: `silver_positions` (16 columns, \~3M rows) → Output: `gold_possession_zones` (16 columns, \~9K sequences)

| Step | Cell | Action | Columns Added / Modified |
| --- | --- | --- | --- |
| 0 | 2 | Load `silver_positions` from Delta table | 16 input columns loaded |
| 1 | 8 | Filter to team rows only (`team_id != "BALL"`). Propagate opponent metrics via window per frame: `_opp_frame_score`, `_opp_possession_zone`, `_opp_ball_distance_target`, `_opp_offside_line`, `_opp_offside_line_perc`, `_opp_team_id`. | 6 temp `_opp_*` columns added via window |
| 2 | 8 | Filter to `has_possession == True` (possessing team rows only). | Rows reduced to possessing team frames only |
| 3 | 8 | Gaps-and-islands: `row_num` (per match+section, ordered by frame_id) minus `row_num_zone` (per match+section+zone+team, ordered by frame_id) = `group_id`. Breaks when `possession_zone` or `team_id` changes. | `group_id` computed (temp) |
| 4 | 8 | Aggregate per group (`match_id`, `game_section`, `team_id`, `possession_zone`, `group_id`): `start_frame`, `end_frame`, `num_frames`, `active_frames`, `interruption_frames`, `cumulative_score`, opponent metrics, `min_ball_distance_target`, `avg_ball_speed`, `avg_offside_line`, `avg_offside_line_perc`, `opponent_id`. | 17 aggregated columns |
| 5 | 8 | Add `duration_sec` (`num_frames / 25`), `duration_min`, `cumulative_time` (running sum of duration per match+section, formatted `MM:SS`). | `duration_sec`, `duration_min`, `cumulative_time` added |
| 6 | 8 | Add `possession_id` via lag: increment when `team_id` changes between consecutive sequences (running sum of `_is_new_poss`). | `possession_id` added |
| 7 | 8 | Add `stage_id` via `row_number()` per match+section ordered by `start_frame`. | `stage_id` added |
| 8 | 8 | Build `team_metrics` struct (possession_zone, cumulative_score, min_ball_distance_target, avg_ball_speed, avg_offside_line, avg_offside_line_perc) and `opponent_metrics` struct (same fields from opponent). | `team_metrics`, `opponent_metrics` structs added |

**Output schema (16 columns):**

| # | Column | Type | Source |
| --- | --- | --- | --- |
| 1 | `possession_id` | int | Step 6 — lag-based, increments on team change |
| 2 | `stage_id` | int | Step 7 — row_number per match+section |
| 3 | `match_id` | string | Group key |
| 4 | `game_section` | string | Group key |
| 5 | `cumulative_time` | string | Step 5 — running duration `MM:SS` |
| 6 | `team_id` | string | Group key — possessing team |
| 7 | `opponent_id` | string | Step 4 — from opponent window |
| 8 | `start_frame` | long | Step 4 — min frame_id in group |
| 9 | `end_frame` | long | Step 4 — max frame_id in group |
| 10 | `num_frames` | long | Step 4 — count distinct frames |
| 11 | `duration_sec` | double | Step 5 — num_frames / 25 FPS |
| 12 | `duration_min` | double | Step 5 — duration_sec / 60 |
| 13 | `active_frames` | long | Step 4 — play_state = active count |
| 14 | `interruption_frames` | long | Step 4 — play_state = interruption count |
| 15 | `team_metrics` | struct | Step 8 — possession_zone, cumulative_score, min_ball_distance_target, avg_ball_speed, avg_offside_line, avg_offside_line_perc |
| 16 | `opponent_metrics` | struct | Step 8 — opponent's mirrored metrics |

In [0]:
silver_df.groupBy("match_id","play_state").count().show()

In [0]:
# ── DEPRECATED: play_state sequences (active/interruption runs) ──
# This was an early prototype grouping consecutive frames by play_state.
# Superseded by gold_possessions_all (cell 6) and gold_possession_zones (cell 8).
# Table `gold_play_state` is not used in the pipeline.
#
# Original code preserved below for reference.

from pyspark.sql import functions as F
silver_df.groupBy("match_id").agg(
    F.min("frame_id").alias("min_frame"),
    F.max("frame_id").alias("max_frame"),
    F.count("*").alias("total_rows"),
    F.countDistinct("frame_id").alias("distinct_frames"),
).orderBy("match_id").show(truncate=False)

print("=== play_state distribution per match ===")
silver_df.groupBy("match_id", "play_state").count().orderBy("match_id", "count", ascending=False).show(truncate=False)

# Gaps-and-islands: group consecutive frames with the same play_state into sequences
match_frames = silver_df \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id").orderBy("frame_id"))) \
    .withColumn("row_num_state", F.row_number().over(Window.partitionBy("match_id", "play_state").orderBy("frame_id"))) \
    .withColumn("group_id", F.col("row_num") - F.col("row_num_state"))

# Each group is a consecutive run of the same play_state
# Convert frame span to approximate minutes (common tracking rate is 25 fps)
FPS = 25
match_frames.groupBy("match_id", "game_section", "play_state", "group_id") \
    .agg(
        F.min("frame_id").alias("start_frame"),
        F.max("frame_id").alias("end_frame"),
        F.countDistinct("frame_id").alias("num_frames"),
    ) \
    .withColumn("duration_sec", F.col("num_frames") / F.lit(FPS)) \
    .withColumn("duration_min", F.round(F.col("duration_sec") / 60, 2)) \
    .withColumn("cumulative_time", F.concat(
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) / 60).cast("int").cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) % 60).cast("int").cast("string"), 2, "0")
    )) \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id").orderBy("start_frame"))) \
    .select("row_num", "match_id", "game_section", "play_state", "group_id", "start_frame", "end_frame", "num_frames", "duration_sec", "duration_min", "cumulative_time") \
    .orderBy("match_id", "start_frame")

gold_df = match_frames.groupBy("match_id", "game_section", "play_state", "group_id") \
    .agg(
        F.min("frame_id").alias("start_frame"),
        F.max("frame_id").alias("end_frame"),
        F.countDistinct("frame_id").alias("num_frames"),
    ) \
    .withColumn("duration_sec", F.col("num_frames") / F.lit(FPS)) \
    .withColumn("duration_min", F.round(F.col("duration_sec") / 60, 2)) \
    .withColumn("cumulative_time", F.concat(
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) / 60).cast("int").cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) % 60).cast("int").cast("string"), 2, "0")
    )) \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id").orderBy("start_frame"))) \
    .select("row_num", "match_id", "game_section", "play_state", "group_id", "start_frame", "end_frame", "num_frames", "duration_sec", "duration_min", "cumulative_time") \
    .orderBy("match_id", "start_frame")

gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`bundesliga-2022-2023`.batch.gold_play_state")

print("Saved to gold table: gold_play_state")
gold_df.show(truncate=False)

In [0]:
# ── DEPRECATED: possession sequences from active play only ──
# This was an early prototype filtering to active play only.
# Superseded by gold_possessions_all (cell 6) which includes ALL play states.
# Table `gold_possessions` is not used in the pipeline.
#
# Original code preserved below for reference.

from pyspark.sql import functions as F
possession_frames = silver_df \
    .filter(F.col("play_state") == "active") \
    .filter(F.col("has_possession") == True) \
    .filter(F.col("team_id") != "BALL")

# Gaps-and-islands: detect consecutive frames where the same team has possession
# Break streaks when team_id or game_section changes
possession_frames = possession_frames \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id", "game_section").orderBy("frame_id"))) \
    .withColumn("row_num_team", F.row_number().over(Window.partitionBy("match_id", "game_section", "team_id").orderBy("frame_id"))) \
    .withColumn("group_id", F.col("row_num") - F.col("row_num_team"))

# Aggregate each possession sequence
possession_sequences = possession_frames.groupBy("match_id", "game_section", "team_id", "group_id") \
    .agg(
        F.min("frame_id").alias("start_frame"),
        F.max("frame_id").alias("end_frame"),
        F.countDistinct("frame_id").alias("num_frames"),
        F.min("timestamp").alias("start_time"),
        F.max("timestamp").alias("end_time"),
    ) \
    .withColumn("duration_sec", F.col("num_frames") / F.lit(FPS)) \
    .withColumn("duration_min", F.round(F.col("duration_sec") / 60, 2)) \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id").orderBy("start_frame"))) \
    .select("row_num", "match_id", "game_section", "team_id", "group_id", "start_frame", "end_frame", "num_frames", "duration_sec", "duration_min", "start_time", "end_time") \
    .orderBy("match_id", "start_frame")

possession_sequences.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`bundesliga-2022-2023`.batch.gold_possessions")

print("Saved to gold table: gold_possessions")
print(f"Total possession sequences: {possession_sequences.count()}")
possession_sequences.show(30, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

FPS = 25

# Filter to team rows with possession (ALL play states, not just active)
possession_frames_all = silver_df \
    .filter(F.col("has_possession") == True) \
    .filter(F.col("team_id") != "BALL")

# Gaps-and-islands: detect consecutive frames where the same team has possession
# Break streaks only when team_id or game_section changes (NOT on play_state)
possession_frames_all = possession_frames_all \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id", "game_section").orderBy("frame_id"))) \
    .withColumn("row_num_team", F.row_number().over(Window.partitionBy("match_id", "game_section", "team_id").orderBy("frame_id"))) \
    .withColumn("group_id", F.col("row_num") - F.col("row_num_team"))

# Aggregate each possession sequence (no play_state split, no start/end time)
possession_sequences_all = possession_frames_all.groupBy("match_id", "game_section", "team_id", "group_id") \
    .agg(
        F.min("frame_id").alias("start_frame"),
        F.max("frame_id").alias("end_frame"),
        F.countDistinct("frame_id").alias("num_frames"),
    ) \
    .withColumn("duration_sec", F.col("num_frames") / F.lit(FPS)) \
    .withColumn("duration_min", F.round(F.col("duration_sec") / 60, 2)) \
    .withColumn("cumulative_time", F.concat(
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) / 60).cast("int").cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) % 60).cast("int").cast("string"), 2, "0")
    )) \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id").orderBy("start_frame"))) \
    .select("row_num", "match_id", "game_section", "team_id", "group_id", "start_frame", "end_frame", "num_frames", "duration_sec", "duration_min", "cumulative_time") \
    .orderBy("match_id", "start_frame")

# possession_sequences_all.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`bundesliga-2022-2023`.batch.gold_possessions_all")
# Note: Table is managed by the SDP pipeline as a materialized view. Saving from notebook is disabled.

print("Saved to gold table: gold_possessions_all")
print(f"Total possession sequences: {possession_sequences_all.count()}")
possession_sequences_all.show(30, truncate=False)

In [0]:
# ── DEPRECATED: 9-cell pitch zone mapping demo ──
# This was an exploratory demo of the pitch_zone classification.
# The logic now lives in bronze_positions (pitch_zone, zone_id columns).
# This cell is not used in any pipeline transformation.
#
# Original code preserved below for reference.

from pyspark.sql import functions as F
PITCH_X = 105.0
PITCH_Y = 68.0

# Build a Spark expression that maps (x, y) -> one of 9 pitch zones
# X axis: first-third (0-35), second-third (35-70), final-third (70-105)
# Y axis: left (0-22.67), centre (22.67-45.33), right (45.33-68)
def pitch_zone(x_col, y_col, pitch_x=PITCH_X, pitch_y=PITCH_Y):
    x_third = (
        F.when(x_col < pitch_x / 3, F.lit("first-third"))
         .when(x_col < 2 * pitch_x / 3, F.lit("second-third"))
         .otherwise(F.lit("final-third"))
    )
    y_zone = (
        F.when(y_col < pitch_y / 3, F.lit("left"))
         .when(y_col < 2 * pitch_y / 3, F.lit("centre"))
         .otherwise(F.lit("right"))
    )
    return F.concat(x_third, F.lit("_"), y_zone)

# Numeric zone id 1-9 via create_map
zone_map = F.create_map(
    F.lit("first-third_left"),    F.lit(1),
    F.lit("first-third_centre"),  F.lit(2),
    F.lit("first-third_right"),   F.lit(3),
    F.lit("second-third_left"),  F.lit(4),
    F.lit("second-third_centre"), F.lit(5),
    F.lit("second-third_right"),  F.lit(6),
    F.lit("final-third_left"),   F.lit(7),
    F.lit("final-third_centre"), F.lit(8),
    F.lit("final-third_right"),   F.lit(9),
)

# Demo: apply to silver_positions players (raw + normalized coords)
zones_df = silver_df.filter(F.col("team_id") != "BALL") \
    .withColumn("player", F.explode("players")) \
    .withColumn("pitch_zone", pitch_zone(F.col("player.x"), F.col("player.y"))) \
    .withColumn("pitch_zone_norm", pitch_zone(F.col("player.x_norm"), F.col("player.y_norm"))) \
    .withColumn("zone_id", zone_map[F.col("pitch_zone")])

print("=== Zone distribution (raw x/y) ===")
zones_df.groupBy("pitch_zone").count().orderBy("pitch_zone").show(truncate=False)

print("=== Zone distribution (x_norm/y_norm) ===")
zones_df.groupBy("pitch_zone_norm").count().orderBy("pitch_zone_norm").show(truncate=False)

print("=== Sample ===")
zones_df.select("match_id", "frame_id", "team_id", "player.person_id",
                 "player.x", "player.y", "pitch_zone", "zone_id") \
    .show(20, truncate=False)

In [0]:
# ── Possession zone sequences: gaps-and-islands on possession_zone ──
# Similar to play_state consecutive sequences, but now we track consecutive frames
# where the ball stays in the same possession_zone with the same team in possession.
# Break streaks when: possession_zone changes, team_id changes, or game_section changes.
# Includes ALL play states (active + interruption) — play_state is included in output.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

FPS = 25

# All team rows (not BALL) — needed to get opponent's frame_score and possession_zone
all_team_frames = silver_df.filter(F.col("team_id") != "BALL")

# For each frame, propagate the opponent's frame_score and possession_zone via window
w_frame = Window.partitionBy("match_id", "frame_id")
all_team_frames = all_team_frames \
    .withColumn("_opp_frame_score",
        F.sum(F.when(F.col("has_possession") == False, F.col("frame_score")).otherwise(F.lit(0))).over(w_frame)
    ) \
    .withColumn("_opp_possession_zone",
        F.max(F.when(F.col("has_possession") == False, F.col("possession_zone")).otherwise(F.lit(None))).over(w_frame)
    ) \
    .withColumn("_opp_ball_distance_target",
        F.max(F.when(F.col("has_possession") == False, F.col("ball_distance_target")).otherwise(F.lit(None))).over(w_frame)
    ) \
    .withColumn("_opp_offside_line",
        F.max(F.when(F.col("has_possession") == False, F.col("offside_line")).otherwise(F.lit(None))).over(w_frame)
    ) \
    .withColumn("_opp_offside_line_perc",
        F.max(F.when(F.col("has_possession") == False, F.col("offside_line_perc")).otherwise(F.lit(None))).over(w_frame)
    ) \
    .withColumn("_opp_team_id",
        F.max(F.when(F.col("has_possession") == False, F.col("team_id")).otherwise(F.lit(None))).over(w_frame)
    )

# Now filter to only the possessing team for gaps-and-islands
zone_frames = all_team_frames \
    .filter(F.col("has_possession") == True)

# Gaps-and-islands: break when possession_zone or team_id changes
zone_frames = zone_frames \
    .withColumn("row_num", F.row_number().over(Window.partitionBy("match_id", "game_section").orderBy("frame_id"))) \
    .withColumn("row_num_zone", F.row_number().over(Window.partitionBy("match_id", "game_section", "possession_zone", "team_id").orderBy("frame_id"))) \
    .withColumn("group_id", F.col("row_num") - F.col("row_num_zone"))

# Aggregate each zone sequence
zone_sequences = zone_frames.groupBy("match_id", "game_section", "team_id", "possession_zone", "group_id") \
    .agg(
        F.min("frame_id").alias("start_frame"),
        F.max("frame_id").alias("end_frame"),
        F.countDistinct("frame_id").alias("num_frames"),
        F.sum(F.when(F.col("play_state") == "active", 1).otherwise(0)).alias("active_frames"),
        F.sum(F.when(F.col("play_state") == "interruption", 1).otherwise(0)).alias("interruption_frames"),
        F.round(F.sum("frame_score"), 4).alias("cumulative_score"),
        F.round(F.sum("_opp_frame_score"), 4).alias("opponent_cumulative_score"),
        F.max("_opp_possession_zone").alias("opponent_possession_zone"),
        F.min("ball_distance_target").alias("min_ball_distance_target"),
        F.min("_opp_ball_distance_target").alias("opponent_min_ball_distance_target"),
        F.round(F.avg(F.when(F.expr("get(players, 0).distance") > 0, F.expr("get(players, 0).speed"))), 2).alias("avg_ball_speed"),
        F.round(F.avg("offside_line"), 2).alias("avg_offside_line"),
        F.round(F.avg("offside_line_perc"), 2).alias("avg_offside_line_perc"),
        F.round(F.avg("_opp_offside_line"), 2).alias("opponent_avg_offside_line"),
        F.round(F.avg("_opp_offside_line_perc"), 2).alias("opponent_avg_offside_line_perc"),
        F.max("_opp_team_id").alias("opponent_id"),
    ) \
    .withColumn("duration_sec", F.col("num_frames") / F.lit(FPS)) \
    .withColumn("duration_min", F.round(F.col("duration_sec") / 60, 2)) \
    .withColumn("cumulative_time", F.concat(
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) / 60).cast("int").cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.sum(F.col("duration_sec")).over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow)) % 60).cast("int").cast("string"), 2, "0")
    )) \
    .withColumn("_prev_team", F.lag("team_id").over(Window.partitionBy("match_id", "game_section").orderBy("start_frame"))) \
    .withColumn("_is_new_poss", F.when(F.col("_prev_team").isNull() | (F.col("team_id") != F.col("_prev_team")), 1).otherwise(0)) \
    .withColumn("possession_id", F.sum("_is_new_poss").over(Window.partitionBy("match_id", "game_section").orderBy("start_frame").rowsBetween(Window.unboundedPreceding, Window.currentRow))) \
    .drop("_prev_team", "_is_new_poss") \
    .withColumn("stage_id", F.row_number().over(Window.partitionBy("match_id", "game_section").orderBy("start_frame"))) \
    .withColumn("team_metrics", F.struct(
        F.col("possession_zone").alias("possession_zone"),
        F.col("cumulative_score").alias("cumulative_score"),
        F.col("min_ball_distance_target").alias("min_ball_distance_target"),
        F.col("avg_ball_speed").alias("avg_ball_speed"),
        F.col("avg_offside_line").alias("avg_offside_line"),
        F.col("avg_offside_line_perc").alias("avg_offside_line_perc"),
    )) \
    .withColumn("opponent_metrics", F.struct(
        F.col("opponent_possession_zone").alias("possession_zone"),
        F.col("opponent_cumulative_score").alias("cumulative_score"),
        F.col("opponent_min_ball_distance_target").alias("min_ball_distance_target"),
        F.col("opponent_avg_offside_line").alias("avg_offside_line"),
        F.col("opponent_avg_offside_line_perc").alias("avg_offside_line_perc"),
    )) \
    .select("possession_id", "stage_id", "match_id", "game_section", "cumulative_time",
            "team_id", "opponent_id",
            "start_frame", "end_frame", "num_frames", "duration_sec", "duration_min",
            "active_frames", "interruption_frames",
            "team_metrics", "opponent_metrics") \
    .orderBy("match_id", "start_frame")

# zone_sequences.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`bundesliga-2022-2023`.batch.gold_possession_zones")
# Note: Table is managed by the SDP pipeline as a materialized view. Saving from notebook is disabled.

print("Saved to gold table: gold_possession_zones")
print(f"Total zone sequences: {zone_sequences.count()}")
zone_sequences.show(30, truncate=False)

In [0]:
# ── Validate gold_possession_zones ──

from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold_zones = spark.table("`bundesliga-2022-2023`.batch.gold_possession_zones")

print("=== Validation: gold_possession_zones ===")
print(f"  Total sequences: {gold_zones.count():,}")
print(f"  Columns: {len(gold_zones.columns)}")

# 1. stage_id continuity: should start at 1, no gaps per match+section
stage_check = gold_zones.groupBy("match_id", "game_section").agg(
    F.min("stage_id").alias("min_stage"),
    F.max("stage_id").alias("max_stage"),
    F.countDistinct("stage_id").alias("distinct_stages"),
)
bad_stages = stage_check.filter(
    (F.col("min_stage") != 1) |
    (F.col("max_stage") != F.col("distinct_stages"))
)
print(f"\n1. stage_id continuity: {'PASS' if bad_stages.count() == 0 else 'FAIL'}")
if bad_stages.count() > 0:
    print("   Issues:")
    bad_stages.show(truncate=False)
else:
    print("   stage_id starts at 1, no gaps per match+section")

# 2. possession_id continuity: should start at 1, no gaps per match+section
poss_check = gold_zones.groupBy("match_id", "game_section").agg(
    F.min("possession_id").alias("min_poss"),
    F.max("possession_id").alias("max_poss"),
    F.countDistinct("possession_id").alias("distinct_poss"),
)
bad_poss = poss_check.filter(
    (F.col("min_poss") != 1) |
    (F.col("max_poss") != F.col("distinct_poss"))
)
print(f"\n2. possession_id continuity: {'PASS' if bad_poss.count() == 0 else 'FAIL'}")
if bad_poss.count() > 0:
    print("   Issues:")
    bad_poss.show(truncate=False)
else:
    print("   possession_id starts at 1, no gaps per match+section")

# 3. possession_id increments on team change (lag check)
poss_lag = gold_zones.withColumn(
    "_prev_team", F.lag("team_id").over(Window.partitionBy("match_id", "game_section").orderBy("stage_id"))
).withColumn(
    "_expected_new", F.when(F.col("_prev_team").isNull() | (F.col("team_id") != F.col("_prev_team")), 1).otherwise(0)
).withColumn(
    "_expected_poss_id", F.sum("_expected_new").over(Window.partitionBy("match_id", "game_section").orderBy("stage_id").rowsBetween(Window.unboundedPreceding, Window.currentRow))
)
mismatch = poss_lag.filter(F.col("_expected_poss_id") != F.col("possession_id")).count()
print(f"\n3. possession_id lag check: {'PASS' if mismatch == 0 else 'FAIL'}")
print(f"   Mismatches: {mismatch}")

# 4. No nulls in key columns
key_cols = ["possession_id", "stage_id", "match_id", "game_section", "team_id", "opponent_id",
            "start_frame", "end_frame", "num_frames", "duration_sec"]
null_counts = gold_zones.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key_cols])
null_row = null_counts.collect()[0]
has_nulls = any(v > 0 for v in null_row)
print(f"\n4. Null check (key columns): {'PASS' if not has_nulls else 'FAIL'}")
if has_nulls:
    for c, v in zip(key_cols, null_row):
        if v > 0:
            print(f"   {c}: {v} nulls")
else:
    print("   No nulls in any key column")

# 5. Row count per match
print(f"\n5. Sequences per match:")
gold_zones.groupBy("match_id").agg(
    F.count("*").alias("num_sequences"),
    F.countDistinct("possession_id").alias("distinct_possessions"),
    F.round(F.avg("duration_sec"), 2).alias("avg_duration_sec"),
).orderBy("match_id").show(truncate=False)

# 6. active + interruption = num_frames
frame_check = gold_zones.filter(F.col("active_frames") + F.col("interruption_frames") != F.col("num_frames"))
print(f"\n6. active + interruption = num_frames: {'PASS' if frame_check.count() == 0 else 'FAIL'}")
print(f"   Mismatches: {frame_check.count()}")

In [0]:
# ── Zone sequence analysis ──

from pyspark.sql import functions as F

gold_zones = spark.table("`bundesliga-2022-2023`.batch.gold_possession_zones")

print(f"=== Zone sequences summary ===")
print(f"  Total sequences: {gold_zones.count():,}")
print(f"  Avg frames per sequence: {gold_zones.agg(F.avg('num_frames')).collect()[0][0]:.1f}")
print(f"  Avg duration (sec): {gold_zones.agg(F.avg('duration_sec')).collect()[0][0]:.2f}")

print(f"\n=== Sequences per zone ===")
gold_zones.groupBy(F.col("team_metrics.possession_zone")) \
    .agg(
        F.count("*").alias("num_sequences"),
        F.sum("num_frames").alias("total_frames"),
        F.round(F.avg("duration_sec"), 2).alias("avg_duration_sec"),
        F.max("duration_sec").alias("max_duration_sec"),
    ) \
    .orderBy(F.col("team_metrics.possession_zone")) \
    .show(truncate=False)

print(f"\n=== Sample: team_metrics & opponent_metrics ===")
display(
    gold_zones.select("possession_id", "team_id", "team_metrics", "opponent_metrics")
    .limit(10)
)

print(f"\n=== Sequences per team per zone (top 20 by total frames) ===")
gold_zones.groupBy("team_id", F.col("team_metrics.possession_zone")) \
    .agg(
        F.count("*").alias("num_sequences"),
        F.sum("num_frames").alias("total_frames"),
        F.round(F.avg("duration_sec"), 2).alias("avg_duration_sec"),
    ) \
    .orderBy(F.desc("total_frames")) \
    .show(20, truncate=False)

In [0]:
# ── Cumulative score distribution analysis ──

from pyspark.sql import functions as F

gold = spark.table("`bundesliga-2022-2023`.batch.gold_possession_zones")

# Extract cumulative_score from team_metrics struct
stats = gold.select(
    "possession_id", "match_id", "team_id",
    F.col("team_metrics.possession_zone").alias("possession_zone"),
    "num_frames", "duration_sec",
    F.col("team_metrics.cumulative_score").alias("cum_score"),
    F.col("opponent_metrics.cumulative_score").alias("opp_cum_score"),
)

# Overall distribution
total = stats.count()
positive = stats.filter(F.col("cum_score") > 0).count()
negative = stats.filter(F.col("cum_score") < 0).count()
zero = stats.filter(F.col("cum_score") == 0).count()

print("=== Cumulative score distribution (team_metrics) ===")
print(f"  Total sequences:      {total:,}")
print(f"  Positive (approach): {positive:,} ({100*positive/total:.1f}%)")
print(f"  Negative (retreat):  {negative:,} ({100*negative/total:.1f}%)")
print(f"  Zero:                {zero:,} ({100*zero/total:.1f}%)")

# Same for opponent
opp_pos = stats.filter(F.col("opp_cum_score") > 0).count()
opp_neg = stats.filter(F.col("opp_cum_score") < 0).count()
opp_zero = stats.filter(F.col("opp_cum_score") == 0).count()

print(f"\n=== Cumulative score distribution (opponent_metrics) ===")
print(f"  Positive (approach): {opp_pos:,} ({100*opp_pos/total:.1f}%)")
print(f"  Negative (retreat):  {opp_neg:,} ({100*opp_neg/total:.1f}%)")
print(f"  Zero:                {opp_zero:,} ({100*opp_zero/total:.1f}%)")

# Break down by zone
zone_stats = stats.groupBy("possession_zone").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("cum_score") > 0, 1).otherwise(0)).alias("positive"),
    F.sum(F.when(F.col("cum_score") < 0, 1).otherwise(0)).alias("negative"),
    F.sum(F.when(F.col("cum_score") == 0, 1).otherwise(0)).alias("zero"),
    F.round(F.avg("cum_score"), 4).alias("avg_cum_score"),
    F.round(F.max("cum_score"), 4).alias("max_cum_score"),
    F.round(F.min("cum_score"), 4).alias("min_cum_score"),
).withColumn("pct_positive", F.round(F.col("positive") / F.col("total") * 100, 1)).orderBy("possession_zone")

print(f"\n=== Cumulative score by possession zone ===")
zone_stats.show(truncate=False)

# Top 10 most positive and most negative
print(f"\n=== Top 10 most positive cumulative_score (strongest approach) ===")
stats.orderBy(F.desc("cum_score")).select(
    "possession_id", "match_id", "team_id", "possession_zone", "num_frames", "duration_sec", "cum_score"
).show(10, truncate=False)

print(f"\n=== Top 10 most negative cumulative_score (strongest retreat) ===")
stats.orderBy("cum_score").select(
    "possession_id", "match_id", "team_id", "possession_zone", "num_frames", "duration_sec", "cum_score"
).show(10, truncate=False)